# Partie 3 : mouvement soudain du patient

In [1]:
import numpy as np
from scipy.ndimage import rotate
from kspace_fig import (load_kspace, ifft2c, fft2c, make_frame, write_figure, write_static,
                        write_motion_anim, TR_MS)

K = load_kspace()
N0, N1 = K.shape
c1 = N1 // 2
I_ref = ifft2c(K)

def tourne(angle):
    # scipy ne tourne pas les complexes : Re et Im séparément (la rotation est linéaire)
    return (rotate(I_ref.real, angle, reshape=False, order=1)
            + 1j * rotate(I_ref.imag, angle, reshape=False, order=1))

K_rot = fft2c(tourne(20))

def mouvement(split):
    # colonnes < split acquises avant le mouvement, les autres après
    Km = K.copy()
    Km[:, split:] = K_rot[:, split:]
    return Km

In [2]:
def trait(split):
    return [{"type": "line", "x0": split - 0.5, "x1": split - 0.5, "y0": -0.5, "y1": N0 - 0.5,
             "line": {"color": "rgb(220,50,47)", "width": 2}}]

frames = [make_frame(K, "aucun mouvement")]
for s in range(1, N1):
    t = f"{s * TR_MS / 1000:.3f}".replace(".", ",")
    frames.append(make_frame(mouvement(s), f"ligne {s} · t = {t} s", kshapes=trait(s)))

write_figure("../figures/fig-mouvement.html", frames, slider_name="Instant du mouvement",
             title_k="k hybride (avant | après)", active=c1)
write_static("../img/fig-mouvement-static.png",
             [("réf.", K), ("ligne 44", mouvement(c1)), ("ligne 49", mouvement(c1 + 5))])

In [3]:
# mouvement continu : les rotations sont faites dans le navigateur
write_motion_anim("../figures/fig-mouvement-continu.html", K)

T = N1 * TR_MS / 1000
def theta(t, profil, amp=30, centre=10):
    osc = amp * np.sin(2 * np.pi * t / 1.1)
    if profil == "brusque":
        x = np.clip((t - (T / 2 - 0.1)) / 0.2, 0, 1)
        return amp * x * x * (3 - 2 * x)
    if profil == "derive":
        return amp * t / T
    if profil == "hochement":
        return osc
    return np.clip(centre + osc, -amp, amp)

def acquisition(profil):
    Km = np.empty_like(K)
    for n in range(N1):
        Km[:, n] = fft2c(tourne(theta(n * TR_MS / 1000, profil)))[:, n]
    return Km

write_static("../img/fig-mouvement-continu-static.png",
             [(p, acquisition(p)) for p in ["brusque", "derive", "hochement", "asymétrique"]])